In [ ]:
# =============================================================================
# TASK 1: Graph Concepts & State Design
# =============================================================================
# LANGGRAPH CORE BUILDING BLOCKS:
#
# 1. StateGraph — Container for your workflow (like a blank flowchart)
# 2. State — Shared TypedDict that all nodes read from and write to
# 3. Node — A function that takes state, does work, returns updated fields
# 4. Edge — Transition to next node (add_edge "plan" -> "execute")
# 5. Conditional Edge — Smart transition: "if score >= 7 go here, else go there"
# 6. Entry Point — Where execution starts (set_entry_point)
# 7. END — Special node that stops the graph
# =============================================================================

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator

# --- STATE SCHEMA ---
class ResearchState(TypedDict):
    query: str
    plan: str
    search_results: str
    answer: str
    quality_score: int
    retry_count: int
    max_retries: int

# --- ASCII DIAGRAM OF THE GRAPH ---
diagram = """
+-------------------------------------------------------------------+
|                    RESEARCH ASSISTANT GRAPH                        |
+-------------------------------------------------------------------+
|  +----------+                                                     |
|  |  START   |                                                     |
|  +----+-----+                                                     |
|       v                                                            |
|  +----------+    "Create a plan based on the query"               |
|  |   PLAN   +----------------------------------------+            |
|  +----+-----+                                        |            |
|       v                                              |            |
|  +----------+    "Search and retrieve info"          |            |
|  | EXECUTE  |<-------------------------------+      |            |
|  +----+-----+                                |      |            |
|       v                                      |      |            |
|  +----------+    "Write an answer"           |      |            |
|  | GENERATE +--------------------------------+      |            |
|  +----+-----+                                |      |            |
|       v                                      |      |            |
|  +----------+    "Rate quality 0-10"         |      |            |
|  | CRITIQUE +--------------------------------+      |            |
|  +----+-----+                                      |            |
|       |                                            |            |
|       | if score < 7 AND retries < max             |            |
|       +-------------- RETRY LOOP ------------------+            |
|                                                                    |
|       | if score >= 7 OR retries >= max                           |
|       v                                                            |
|  +----------+                                                     |
|  |   END    |                                                     |
|  +----------+                                                     |
+-------------------------------------------------------------------+
"""
print(diagram)

# --- NODES ---

def plan_node(state):
    """Node 1: Create a plan based on the query."""
    query = state["query"]
    plan = f"Search for '{query}', then summarize findings"
    print(f"  [PLAN] {plan}")
    return {"plan": plan}

def execute_node(state):
    """Node 2: Execute the plan (simulate search)."""
    results = f"Found info about: {state['query']}. LangGraph is a graph-based agent framework."
    print(f"  [EXECUTE] Got results")
    return {"search_results": results}

def generate_node(state):
    """Node 3: Generate an answer from search results."""
    answer = f"Based on research: {state['search_results']}. LangGraph gives branching and self-correction."
    print(f"  [GENERATE] Generated answer")
    return {"answer": answer}

def critique_node(state):
    """Node 4: Critique the answer and assign quality score."""
    retry = state["retry_count"]
    score = 5 if retry == 0 else 8
    print(f"  [CRITIQUE] Score: {score}/10 (attempt #{retry + 1})")
    return {
        "quality_score": score,
        "retry_count": retry + 1,
    }

# --- CONDITIONAL ROUTER ---
def route_after_critique(state):
    score = state["quality_score"]
    retries = state["retry_count"]
    max_retries = state["max_retries"]
    if score >= 7:
        print(f"  [ROUTE] Score {score} >= 7 -> FINISH")
        return "finish"
    elif retries < max_retries:
        print(f"  [ROUTE] Score {score} < 7, retries {retries} < {max_retries} -> RETRY")
        return "retry"
    else:
        print(f"  [ROUTE] Max retries reached -> FINISH")
        return "finish"

# --- BUILD THE GRAPH ---
graph = StateGraph(ResearchState)

graph.add_node("plan", plan_node)
graph.add_node("execute", execute_node)
graph.add_node("generate", generate_node)
graph.add_node("critique", critique_node)

graph.add_edge("plan", "execute")
graph.add_edge("execute", "generate")
graph.add_edge("generate", "critique")

graph.add_conditional_edges("critique", route_after_critique, {
    "retry": "execute",
    "finish": END
})

graph.set_entry_point("plan")
app = graph.compile()

# --- RUN IT ---
print("=" * 60)
result = app.invoke({
    "query": "What is LangGraph?",
    "plan": "", "search_results": "", "answer": "",
    "quality_score": 0, "retry_count": 0, "max_retries": 3,
})

print(f"\nFINAL ANSWER: {result['answer'][:80]}...")
print(f"QUALITY SCORE: {result['quality_score']}/10")
print(f"RETRIES: {result['retry_count']}")

GRAPH BUILT SUCCESSFULLY
  [PLAN] Search for 'What is LangGraph?', then summarize findings
  [EXECUTE] Got results
  [GENERATE] Generated answer
  [CRITIQUE] Score: 5/10 (attempt #1)
  [ROUTE] Score 5 < 7, retries 1 < 3 -> RETRY
  [EXECUTE] Got results
  [GENERATE] Generated answer
  [CRITIQUE] Score: 8/10 (attempt #2)
  [ROUTE] Score 8 >= 7 -> FINISH

FINAL ANSWER: Based on research: Found info about: What is LangGraph?. LangGraph is a graph-ba...
QUALITY SCORE: 8/10
RETRIES: 2
